In [1]:
from fractions import Fraction

# NLP Exercises

# NLP Exercises

Exercises from Juravsky & Martin, _Speech and Language Processing_, Third edn., [draft of 12 January 2025](https://web.stanford.edu/~jurafsky/slp3/old_jan25/).

## Part I: Fundamental Algorithms for NLP

### Chapter 4: Naive Bayes, Text Classification, and Sentiment

**4.1** Assume the following likelihoods for each word being part of a positive or negative movie review, and equal prior probabilities for each class.

| | pos | neg |
| - | - | - |
| I | 0.09 | 0.16 |
| always | 0.07 | 0.06 |
| like | 0.29 | 0.06 |
| foreign | 0.04 | 0.15 |
| films | 0.08 | 0.11 |

What class will Naive bayes assign to the sentence “I always like foreign films.”?

- $$\hat{c}=\textrm{argmax}_{c\in C}P(c|d)=\textrm{argmax}_{c\in C}P(d|c)P(c)$$
- Since the prior probabilities are equal, we can ignore $P(c)$ and move onto the probabilites for the individual words.
- $$\hat{P}(\textrm{I always like foreign films}|\textbf{pos})=0.09\times 0.07\times 0.29\times 0.04\times 0.08$$

In [2]:
print("=",0.09*0.07*0.29*0.04*0.08)

= 5.8464e-06


- $$\hat{P}(\textrm{I always like foreign films}|\textbf{neg})=0.16\times 0.06\times 0.06\times 0.15\times 0.11$$

In [3]:
print("=",0.16*0.06*0.06*0.15*0.11)

= 9.503999999999999e-06


- Naive Bayes will assign the sentence "I always like foreign films" to the class **negative**.

**4.2** Given the following short movie reviews, each labeled with a genre, either comedy or action:
1. fun, couple, love, love  **comedy**
2. fast, furious, shoot **action**
2. couple, fly, fast, fun, fun  **comedy**
2. furious, shoot, shoot, fun  **action**
2. fly, fast, shoot, love  **action**

and a new document D:

> fast, couple, shoot, fly

compute the most likely class for D. Assume a naive Bayes classifier and use add-1 smoothing for the likelihoods.

- The total word type count is 7. The word token count in comedy is 9. The word token count in action is 11.
- $$P(\textbf{comedy})=\frac{2}{5}\qquad P(\textbf{action})=\frac{3}{5}$$
- | $w_i$ | $P(w_i|\textbf{comedy})$ | $P(w_i|\textbf{action})$
  | - | - | - |
  | fast | $\frac{1+1}{9+7}$ | $\frac{2+1}{11+7}$ |
  | couple | $\frac{2+1}{9+7}$ | $\frac{0+1}{11+7}$ |
  | shoot | $\frac{0+1}{9+7}$ | $\frac{4+1}{11+7}$ |
  | fly | $\frac{1+1}{9+7}$ | $\frac{1+1}{11+7}$ |
- $$P(\textrm{fast couple shoot fly}|\textbf{comedy})P(\textbf{comedy})=\frac{1}{8}\times\frac{3}{16}\times\frac{1}{16}\times\frac{1}{8}\times\frac{2}{5}$$

In [4]:
print("=",Fraction(1,8)*Fraction(3,16)*Fraction(1,16)*Fraction(1,8)*Fraction(2,5))

= 3/40960


- $$P(\textrm{fast couple shoot fly}|\textbf{action})P(\textbf{action})=\frac{1}{6}\times\frac{1}{18}\times\frac{5}{18}\times\frac{1}{9}\times\frac{3}{5}$$

In [5]:
print("=",Fraction(1,6)*Fraction(1,18)*Fraction(5,18)*Fraction(1,9)*Fraction(3,5))

= 1/5832


- The most likely class for "fast couple shoot fly" is **action**.

**4.3** Train two models, multinomial naive Bayes and binarized naive Bayes, both with add-1 smoothing, on the following document counts for key sentiment words, with positive or negative class assigned as noted.

| doc | “good” | “poor” | “great” | (class) |
| - | - | - | - | - |
| d1. | 3 | 0 | 3 | pos |
| d2. | 0 | 1 | 2 | pos |
| d3. | 1 | 3 | 0 | neg |
| d4. | 1 | 5 | 2 | neg |
| d5. | 0 | 2 | 0 | neg |

Use both naive Bayes models to assign a class (pos or neg) to this sentence:

> A good, good plot and great characters, but poor acting.

Recall from page 61 that with naive Bayes text classification, we simply ignore (throw out) any word that never occurred in the training document. (We
don’t throw out words that appear in some classes but not others; that’s what add-one smoothing is for.) Do the two models agree or disagree?

- I don't see how I'm supposed to do this without the word counts for each of the documents. For the purposes of this exercise, I will assume that each document has length 10.

In [6]:
# Store the results from the above table in a dictionary.
multinom = {1: {'pos': True,
                'counts': {'good': 3,
                           'poor': 0,
                           'great': 3}},
            2: {'pos': True,
                'counts': {'good': 0,
                           'poor': 1,
                           'great': 2}},
            3: {'pos': False,
                'counts': {'good': 1,
                           'poor': 3,
                           'great': 0}},
            4: {'pos': False,
                'counts': {'good': 1,
                           'poor': 5,
                           'great': 2}},
            5: {'pos': False,
                'counts': {'good': 0,
                           'poor': 2,
                           'great': 0}}}
# For the binarized case, we clip the counts at 1.
binar = {1: {'pos': True,
             'counts': {'good': 1,
                        'poor': 0,
                        'great': 1}},
         2: {'pos': True,
             'counts': {'good': 0,
                        'poor': 1,
                        'great': 1}},
         3: {'pos': False,
             'counts': {'good': 1,
                        'poor': 1,
                        'great': 0}},
         4: {'pos': False,
             'counts': {'good': 1,
                        'poor': 1,
                        'great': 1}},
         5: {'pos': False,
             'counts': {'good': 0,
                        'poor': 1,
                        'great': 0}}}
# Return P(c|d) for each c.
def posterior(text:str, counts_dict:dict) -> dict:
    tokens = [w.strip(",.!\"'?").lower() for w in text.split(" ")]
    poss = {k:v['counts'] for k,v in counts_dict.items() if v['pos']}
    negs = {k:v['counts'] for k,v in counts_dict.items() if not v['pos']}
    output = {}
    for i,cls in enumerate([poss,negs]):
        # Start with the prior probability.
        res = len(cls.keys())/(len(poss.keys())+len(negs.keys()))
        # Multiply by each likelihood in turn.
        for token in tokens:
            for d in cls.values():
                if token in d.keys():
                    numerator = d[token]+1
                    denominator = 11 #10 tokens in each sentence + 1
                    res *= Fraction(numerator,denominator)
        if i==0:
            output['pos'] = res
        else:
            output['neg'] = res
    return output

In [7]:
test_s = 'A good, good plot and great characters, but poor acting.'
print(f"""Multinomial naive Bayes
-----------------------
Positive probability:\t{posterior(test_s, multinom)['pos']}
Negative probability:\t{posterior(test_s, multinom)['neg']}

Binarized naive Bayes
---------------------
Positive probability:\t{posterior(test_s, binar)['pos']}
Negative probability:\t{posterior(test_s, binar)['neg']}""")

Multinomial naive Bayes
-----------------------
Positive probability:	7.165553360021506e-07
Negative probability:	6.60712863604196e-10

Binarized naive Bayes
---------------------
Positive probability:	5.971294466684589e-08
Negative probability:	4.894169360031083e-11


- The two models agree on the classification of the sentence.